# Can we predict `passed_quiz`?
Protocol: `protocol.yaml`. Fixed holdout (`holdout.csv`), 5-fold GroupKFold on `student_key` inside the training rows, model chosen by mean CV ROC AUC, holdout scored once.

**Feature timing (columns.csv).** `session_minutes` (length of the whole session) and `hints_used` (hints opened during the session) are session totals. The quiz *closes* the session, so these are only final once the quiz is over; they are not known at prediction time and are excluded. Used: `enrol_year`, `school_code` (categorical), `prior_score` (placement test, recorded earlier). `student_key` is excluded as an entity key.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

df = pd.read_csv('data.csv')
hold_rows = pd.read_csv('holdout.csv')['row'].to_numpy()
assert hashlib.sha256(open('data.csv', 'rb').read()).hexdigest() == '5a0b0221ed641db40322e1a95cae32cc67d3525ecb500813f95d6c109b2a7409'
assert len(df) == 12453 and len(hold_rows) == 2465
is_hold = np.zeros(len(df), bool); is_hold[hold_rows] = True
train_idx = np.flatnonzero(~is_hold); hold_idx = np.flatnonzero(is_hold)
TARGET = 'passed_quiz'
NUM = ['enrol_year', 'prior_score']; CAT = ['school_code']
FEATURES = NUM + CAT
X, y, g = df[FEATURES], df[TARGET], df['student_key']
print(len(train_idx), len(hold_idx), y.iloc[train_idx].mean())
assert not set(g.iloc[train_idx]) & set(g.iloc[hold_idx])

## Candidates (all named before any scoring)

In [ ]:
def prep(scale):
    num = Pipeline([('imp', SimpleImputer(strategy='median'))] + ([('sc', StandardScaler())] if scale else []))
    return ColumnTransformer([('num', num, NUM), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT)])
candidates = {
    'baseline_prior': Pipeline([('p', prep(False)), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'logreg_C0.1': Pipeline([('p', prep(True)), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'hgb_shallow': Pipeline([('p', prep(False)), ('m', HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=150, random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows (preprocessing sits inside each pipeline, so it is fitted per fold)

In [ ]:
gkf = GroupKFold(n_splits=5)
folds = list(gkf.split(train_idx, groups=g.iloc[train_idx]))
cv_folds = [[train_idx[a].tolist(), train_idx[b].tolist()] for a, b in folds]
cv_scores = {}
for name, model in candidates.items():
    s = []
    for a, b in folds:
        m = clone(model).fit(X.iloc[train_idx[a]], y.iloc[train_idx[a]])
        s.append(roc_auc_score(y.iloc[train_idx[b]], m.predict_proba(X.iloc[train_idx[b]])[:, 1]))
    cv_scores[name] = s
pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in cv_scores.items()}).T

## Choose by mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(X.iloc[train_idx], y.iloc[train_idx])
holdout_auc = roc_auc_score(y.iloc[hold_idx], final.predict_proba(X.iloc[hold_idx])[:, 1])
print(chosen, 'CV', np.mean(cv_scores[chosen]), 'holdout ROC AUC', holdout_auc)

In [ ]:
evidence = {
    'features': FEATURES,
    'candidates': list(candidates),
    'cv_folds': cv_folds,
    'cv_scores': {k: [float(x) for x in v] for k, v in cv_scores.items()},
    'chosen': chosen,
    'holdout_rows': hold_idx.tolist(),
    'holdout_scores': {chosen: float(holdout_auc)},
}
json.dump(evidence, open('evidence.json', 'w'))